## Implement Semantic Search on Lakebase

### Installing Utilities and Libraries

In [ ]:
%pip install psycopg[binary]==3.3.4 psycopg_pool==3.3.1 "databricks-sdk>=0.89.0" langchain-community==0.4.1 databricks-openai==0.17.1

### Restart the Python Environment

In [ ]:
dbutils.library.restartPython()

### Setting up the Environment

In [ ]:
from databricks.sdk import WorkspaceClient
import psycopg

# Databricks SDK uses your existing OAuth identity
w = WorkspaceClient()

# Lakebase endpoint resource name
endpoint = (
    "projects/<project-id>/"
    "branches/<branch-id>/"
    "endpoints/<endpoint-id>"
)

# Generate a short-lived OAuth credential
credential = w.postgres.generate_database_credential(
    endpoint=endpoint
)

# Generates the workspace host URL
workspace_host = w.config.host.rstrip("/")

In [ ]:
host = "LAKEBASE_HOTSNAME"
db_name = "LAKEBASE_DATABASE_NAME"
username = "LAKEBASE_USERNAME"
password = credential.token

### Create a Connection Pool

In [ ]:
from psycopg_pool import ConnectionPool

pool = ConnectionPool(
    conninfo=(
        f"host={host} "
        f"dbname={db_name} "
        f"user={username} "
        f"password={password} "
        f"sslmode=require"
    ),
    min_size=2,
    max_size=10
)

pool.wait()

print("Connection pool created successfully")

### Creating the OpenAI Client

In [ ]:
from databricks_openai import DatabricksOpenAI

client = DatabricksOpenAI()

### Create the Embedding Generator Helper Function

In [ ]:
def generate_embeddings(text):

    # OpenAI Request
    completion = client.embeddings.create(
        model="databricks-gte-large-en",
        input=text
    )

    return completion.data[0].embedding

### Generate Vector Embeddings for the User Query

In [ ]:
user_query = "How is GreenSteel reducing emissions?"

query_embedding = generate_embeddings(user_query)

### Implement a Vector Search Query

In [ ]:
search_query = """
SELECT
    ChunkID,
    CompanyName,
    ChunkText,
    ChunkEmbedding <=> %s::vector AS distance
FROM RAG.ESG_Chunks
ORDER BY ChunkEmbedding <=> %s::vector
LIMIT 5
"""

In [ ]:
with pool.connection() as conn:

    with conn.cursor(
        row_factory=dict_row
    ) as cur:

        cur.execute(
            search_query,
            (
                query_embedding,
                query_embedding
            )
        )

        results = cur.fetchall()

for result in results:

    print("company name: {}".format(result["companyname"]))
    print("vector distance: {}".format(result["distance"]))
    print("chunk text: {}".format(result["chunktext"]))
    print("================================")

### Metadata Filtering + Vector Search

In [ ]:
search_query = """
SELECT
    ChunkID,
    CompanyName,
    ChunkText,
    ChunkEmbedding <=> %s::vector AS distance
FROM RAG.ESG_Chunks
WHERE CompanyName = 'GreenSteel Ltd'
ORDER BY ChunkEmbedding <=> %s::vector
LIMIT 5
"""

In [ ]:
with pool.connection() as conn:

    with conn.cursor(
        row_factory=dict_row
    ) as cur:

        cur.execute(
            search_query,
            (
                query_embedding,
                query_embedding
            )
        )

        results = cur.fetchall()

for result in results:

    print("company name: {}".format(result["companyname"]))
    print("vector distance: {}".format(result["distance"]))
    print("chunk text: {}".format(result["chunktext"]))
    print("================================")

### Create a Full Text Search Index

In [ ]:
create_fts_index_query = """CREATE EXTENSION IF NOT EXISTS lakebase_text
ON RAG.ESG_Chunks
USING lakebase_bm25
(
  to_tsvector('english', ChunkText)
)
"""

with pool.connection() as conn:

    with conn.cursor() as cur:

        cur.execute(create_fts_index_query)

    conn.commit()

print("Full Text Search Index Created Successfully")


### Implement Hybrid Search

In [ ]:
search_query = """
SELECT

    ChunkID,
    CompanyName,

    (
        (
            1 -
            (
                ChunkEmbedding <=> %s::vector
            )
        ) * 0.7

        +

        ts_rank(
            to_tsvector(
                'english',
                ChunkText
            ),
            plainto_tsquery(
                'english',
                %s
            )
        ) * 0.3

    ) AS hybrid_score,

    ChunkText

FROM RAG.ESG_Chunks

WHERE

    to_tsvector(
        'english',
        ChunkText
    )

    @@

    plainto_tsquery(
        'english',
        %s
    )

    OR

    (
        ChunkEmbedding <=> %s::vector
    ) < 0.5

ORDER BY hybrid_score DESC

LIMIT 10
"""

In [ ]:
from psycopg.rows import dict_row

with pool.connection() as conn:

    with conn.cursor(
        row_factory=dict_row
    ) as cur:

        cur.execute(
            search_query,
            (
                query_embedding,  # ChunkEmbedding <=> %s::vector
                user_query,    # plainto_tsquery text
                user_query,    # plainto_tsquery text
                query_embedding   # ChunkEmbedding <=> %s::vector
            )
        )

        results = cur.fetchall()

for result in results:

    print(f"Company: {result['companyname']}")
    print(f"Hybrid Score: {result['hybrid_score']:.4f}")
    print("--------------------------------------")
    print(result["chunktext"])
    print("======================================\n")